# CRM메시지 로드

In [ ]:
import json
import gspread
import pandas as pd
from google.cloud import storage

# 1. GCS에서 서비스 계정 정보를 읽어와 Google Sheets API를 인증합니다.
BUCKET_NAME = 'YOUR_GCS_BUCKET_NAME'
KEY_FILE_IN_BUCKET = 'YOUR_SERVICE_ACCOUNT_KEY_FILE.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# gspread 인증
gc = gspread.service_account_from_dict(key_file_dict)


# 2. 이미지와 같은 세로형 매트릭스 구조를 파싱하는 새로운 함수 정의
def get_crm_template_matrix(url, gid):
    """
    B열의 라벨을 기준으로 C, D, E열의 메시지 셋트를 읽어와
    '드랍사유', '시점', '제목', '메시지', 'utm' 컬럼을 가진 DataFrame으로 변환합니다.
    """
    try:
        spreadsheet = gc.open_by_url(url)
        # URL의 gid를 이용해 정확한 탭을 엽니다.
        sheet = spreadsheet.get_worksheet_by_id(gid)
        all_values = sheet.get_all_values()

        # 찾고자 하는 B열의 기준 라벨들
        target_labels = ['드랍 사유', '시점', '제목', '메시지', 'utm']
        parsed_data = {}

        # 시트 전체를 돌며 B열(index 1)이 일치하는 행을 찾아 C, D, E열(index 2~4) 데이터를 추출
        for row in all_values:
            if len(row) > 1:
                label = row[1].strip() # B열 데이터
                if label in target_labels:
                    # C, D, E열의 값을 공백 제거하여 리스트로 저장
                    parsed_data[label] = [cell.strip() for cell in row[2:5]]

        # C, D, E열 데이터를 각각 하나의 레코드(행)로 변환
        records = []
        for i in range(3): # 메시지 세트가 총 3개 (비상주, 예산 초과, 그 외)
            records.append({
                'Drop_Reason': parsed_data.get('드랍 사유', ['', '', ''])[i],
                'Timing': parsed_data.get('시점', ['', '', ''])[i],
                'Subject': parsed_data.get('제목', ['', '', ''])[i],
                'Message': parsed_data.get('메시지', ['', '', ''])[i],
                'UTM': parsed_data.get('utm', ['', '', ''])[i]
            })

        df = pd.DataFrame(records)
        return df

    except gspread.SpreadsheetNotFound:
        print(f"스프레드시트를 찾을 수 없습니다. URL을 확인하세요.")
        raise
    except Exception as e:
        print(f"데이터 파싱 중 오류 발생: {e}")
        raise


# 3. 새로운 URL 및 GID 적용하여 실행
url_new = 'https://docs.google.com/spreadsheets/d/YOUR_GOOGLE_SHEETS_ID_HERE/'
gid_new = YOUR_SHEET_TAB_GID  # 예: 1621498687 형태의 숫자 입력

# 데이터 가져오기
df_crm_msg = get_crm_template_matrix(url_new, gid_new)
df_crm_msg.head()

# Connect SF

In [ ]:
!pip install simple-salesforce

In [ ]:
import requests
import pandas as pd
from simple_salesforce import Salesforce

# 1. 발급받은 정보 입력 (보안 마스킹 완료)
client_id = "YOUR_SALESFORCE_CLIENT_ID"
client_secret = "YOUR_SALESFORCE_CLIENT_SECRET"

# 운영계는 login.salesforce.com / 샌드박스는 test.salesforce.com
# Fastfive 전용 인스턴스 도메인 주소를 플레이스홀더로 대체했습니다.
auth_url = "https://YOUR_ORGANIZATION_DOMAIN.my.salesforce.com/services/oauth2/token"
payload = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret
}

# 변수 초기화 (인증 실패 시를 대비해 미리 빈 데이터프레임으로 생성)
df_ffdrop_1인 = pd.DataFrame()

# 2. 토큰 요청
response = requests.post(auth_url, data=payload)
token_data = response.json()

if response.status_code == 200:
    access_token = token_data.get("access_token")
    instance_url = token_data.get("instance_url")

    # 3. 세일즈포스 API 클라이언트 연결
    sf = Salesforce(instance_url=instance_url, session_id=access_token)

    # 4. 리드 테이블 조건 조회 쿼리문 작성
    query = """
    SELECT Id, Name, Phone, Email, Status, CreatedDate, LastModifiedDate,
           LeadForm__c, CSPreferredArea__c, DeactivateReason__c, IsPersonalInfo__c
    FROM Lead
    WHERE fm_RecordTypeName__c = '공유오피스'
      AND Status = 'Drop'
      AND CSExpectedNumber__c = 1
    """

    print("세일즈포스에서 리드 데이터를 가져오는 중...")
    results = sf.query_all(query)

    # 5. 지정하신 'df_ffdrop_1인' 변수에 데이터 담기
    if results['totalSize'] > 0:
        # 세일즈포스 API 특유의 메타데이터 컬럼(attributes)은 제외하고 깔끔하게 변환
        df_ffdrop_1인 = pd.DataFrame(results['records']).drop(columns=['attributes'], errors='ignore')
        print(f"✅ 성공: 'df_ffdrop_1인'에 총 {len(df_ffdrop_1인)}건의 데이터가 저장되었습니다.")
    else:
        print("❌ 조건에 일치하는 데이터가 세일즈포스에 없습니다. 빈 데이터프레임이 유지됩니다.")

else:
    print("인증 실패:", token_data)

In [ ]:
df_ffdrop_1인.head()

In [ ]:
# 'CreatedDate'와 'LastModifiedDate' 컬럼을 한국 시간으로 변환하는 코드
for col in ['CreatedDate', 'LastModifiedDate']:
    if col in df_ffdrop_1인.columns:
        # 1. 문자열을 날짜형(datetime)으로 변환 (기본 UTC 인식)
        df_ffdrop_1인[col] = pd.to_datetime(df_ffdrop_1인[col])

        # 2. 한국 시간(Asia/Seoul)으로 시차를 바꾸고, 뒤에 붙는 타임존 표시(+09:00)를 깔끔하게 제거
        df_ffdrop_1인[col] = df_ffdrop_1인[col].dt.tz_convert('Asia/Seoul').dt.tz_localize(None)

print("⏰ 날짜 데이터가 한국 시간(KST)으로 변환되었습니다.")

In [ ]:
df_ffdrop_1인.head()

In [ ]:
# 1. 한국 시간(KST) 기준으로 실시간 '어제' 날짜 구하기
# (오늘 날짜에서 1일을 빼고 날짜(Date) 부분만 추출합니다)
yesterday = (pd.Timestamp.now(tz='Asia/Seoul') - pd.Timedelta(days=1)).date()

# 2. LastModifiedDate의 날짜 부분(.dt.date)이 어제와 일치하는 행만 필터링
# 변수명은 말씀하신 df_ffdrop_1인_yesterday 로 지정합니다.
df_ffdrop_1인_yesterday = df_ffdrop_1인[df_ffdrop_1인['LastModifiedDate'].dt.date == yesterday].copy()
df_ffdrop_1인_yesterday.head()

In [ ]:
import pandas as pd

# 1. 한국 시간(KST) 기준 오늘 날짜 구하기 (시간은 00:00:00으로 초기화)
today = pd.Timestamp.now(tz='Asia/Seoul').normalize().tz_localize(None)

# 2. LastModifiedDate를 확실하게 판다스 datetime형으로 변환 및 KST 타임존 정렬
df_ffdrop_1인_yesterday['LastModifiedDate'] = pd.to_datetime(df_ffdrop_1인_yesterday['LastModifiedDate'])
try:
    df_ffdrop_1인_yesterday['LastModifiedDate'] = df_ffdrop_1인_yesterday['LastModifiedDate'].dt.tz_convert('Asia/Seoul').dt.tz_localize(None)
except Exception:
    # 이미 타임존이 없는 상태라면 그냥 타임존 정보만 제거
    df_ffdrop_1인_yesterday['LastModifiedDate'] = df_ffdrop_1인_yesterday['LastModifiedDate'].dt.tz_localize(None)

# 3. 시점 계산: 둘 다 순수 날짜 형태(datetime64)인 상태에서 빼기 연산 수행
# .dt.normalize()를 쓰면 시간(시/분/초)을 떼고 날짜만 비교해 줍니다.
days_diff = (today - df_ffdrop_1인_yesterday['LastModifiedDate'].dt.normalize()).dt.days
df_ffdrop_1인_yesterday['Timing_Calc'] = 'Drop+' + days_diff.astype(str)

# [추가] 4번에서 사용하는 드랍 사유 분류 컬럼 생성
def classify_reason(reason):
    if pd.isna(reason):
        return '그 외'
    reason_str = str(reason).strip()
    if reason_str == '예산 초과':
        return '예산 초과'
    elif '비상주' in reason_str:
        return '비상주'
    else:
        return '그 외'

df_ffdrop_1인_yesterday['Drop_Reason_Calc'] = df_ffdrop_1인_yesterday['DeactivateReason__c'].apply(classify_reason)

# 4. df_crm_msg와 두 가지 조건(사유, 시점)을 기준으로 Left Join 수행
# 결과를 df_ffdrop_1인_7days_joined 변수에 담습니다.
df_ffdrop_1인_yesterday = pd.merge(
    df_ffdrop_1인_yesterday,
    df_crm_msg,
    left_on=['Drop_Reason_Calc', 'Timing_Calc'],  # 세일즈포스 데이터 기준 컬럼
    right_on=['Drop_Reason', 'Timing'],           # 구글 시트 데이터 기준 컬럼
    how='left'
)

print("✅ 시점 및 사유 매핑 후 Left Join이 완료되었습니다.")
print(f"최종 데이터 수: {len(df_ffdrop_1인_yesterday)}건")

In [ ]:
df_ffdrop_1인_yesterday.head()

In [ ]:
import pandas as pd

# ==========================================
# 1. 공통 전처리: 문구 매칭된 대상만 추출 및 전화번호 정제
# ==========================================
# 구글 시트 문구(Drop+1)와 정상 매칭된 데이터만 추출 (Message가 NaN이 아닌 것)
df_valid = df_ffdrop_1인_yesterday[df_ffdrop_1인_yesterday['Message'].notna()].copy()

# 전화번호 하이픈 제거 및 숫자만 추출
df_valid['Phone_Clean'] = df_valid['Phone'].str.replace(r'[^0-9]', '', regex=True)


# ==========================================
# 2. 솔라피(Solapi) 발송 규격 데이터프레임 생성
# ==========================================
df_solapi = pd.DataFrame()
df_solapi['to'] = df_valid['Phone_Clean']          # 수신번호
df_solapi['text'] = df_valid['Message']            # 발송 메시지 본문
df_solapi['subject'] = df_valid['Subject']        # LMS 대체문자 제목
df_solapi['type'] = 'LMS'                         # 장문 고정

# df_solapi['from'] = '0212345678' # 필요한 경우 발신번호 설정

print(f"📱 [솔라피] 전일 드랍 대상자 발송 가공 완료: 총 {len(df_solapi)}건")


# ==========================================
# 3. 빅쿼리(BigQuery) 적재용 데이터프레임 생성
# ==========================================
# 로그 추적용 필수 컬럼 추출
df_bq_load = df_valid[[
    'Id', 'Name', 'Phone_Clean', 'Email', 'Status',
    'CreatedDate', 'LastModifiedDate', 'LeadForm__c',
    'CSPreferredArea__c', 'DeactivateReason__c',
    'Timing_Calc', 'Drop_Reason_Calc', 'Subject', 'Message'
]].copy()

# 실시간 한국 시간 기준으로 '실제 발송 날짜/시각' 기록
df_bq_load['SentAt'] = pd.Timestamp.now(tz='Asia/Seoul').normalize().tz_localize(None)

# 빅쿼리 적재 에러 방지를 위해 날짜 타입을 문자열(String)로 통일
df_bq_load['CreatedDate'] = df_bq_load['CreatedDate'].astype(str)
df_bq_load['LastModifiedDate'] = df_bq_load['LastModifiedDate'].astype(str)
df_bq_load['SentAt'] = df_bq_load['SentAt'].astype(str)

print(f"📊 [빅쿼리] 적재용 Raw 데이터 가공 완료: 총 {len(df_bq_load)}건")

In [ ]:
df_solapi.head()

# 솔라피 발송

In [ ]:
!pip install solapi

In [ ]:
import hmac
import hashlib
import secrets
from datetime import datetime, timezone
import requests
import pandas as pd
from google.oauth2 import service_account
from typing import Dict, Any

# ==========================================
# 0. 인증 및 환경 설정 (보안 마스킹 완료)
# ==========================================
API_KEY = "YOUR_SOLAPI_API_KEY"
API_SECRET = "YOUR_SOLAPI_API_SECRET"
SENDER_NUMBER = "YOUR_SENDER_NUMBER"  # 예: 하이픈 없는 "18335550" 형태

BQ_PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BQ_TARGET_TABLE = "YOUR_DATASET.YOUR_TABLE_NAME"  # 예: "MKT.ff-lead-crm-raw"


# ==========================================
# 1. 솔라피 인증용 자격 증명 함수 정의
# ==========================================
def generate_signature(api_secret: str, date_time: str, salt: str) -> str:
    data = date_time + salt
    return hmac.new(api_secret.encode(), data.encode(), hashlib.sha256).hexdigest()

def create_auth_header(api_key: str, api_secret: str) -> str:
    date_time = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
    salt = secrets.token_hex(16)
    signature = generate_signature(api_secret, date_time, salt)
    return f"HMAC-SHA256 apiKey={api_key}, date={date_time}, salt={salt}, signature={signature}"

def send_message(api_key: str, api_secret: str, message_data: Dict[str, Any]) -> Dict[str, Any]:
    auth_header = create_auth_header(api_key, api_secret)
    headers = {
        'Authorization': auth_header,
        'Content-Type': 'application/json'
    }
    response = requests.post(
        'https://api.solapi.com/messages/v4/send-many/detail',
        json=message_data,
        headers=headers
    )
    response.raise_for_status()
    return response.json()

# ==========================================
# 🚀 운영계(Production) 라이브 발송 및 적재 공정
# ==========================================
if __name__ == "__main__":
    # 환경 변수 재바인딩부 마스킹
    API_KEY = "YOUR_SOLAPI_API_KEY"
    API_SECRET = "YOUR_SOLAPI_API_SECRET"
    SENDER_NUMBER = "YOUR_SENDER_NUMBER"

    BQ_PROJECT_ID = "YOUR_GCP_PROJECT_ID"
    BQ_TARGET_TABLE = "YOUR_DATASET.YOUR_TABLE_NAME"

    # 1. 실제 고객 대상 솔라피 페이로드 생성
    messages_payload = []
    for idx, row in df_solapi.iterrows():  # df_solapi 기반 라이브 발송
        messages_payload.append({
            "to": row['to'],
            "from": SENDER_NUMBER,
            "subject": row['subject'],
            "text": row['text'],
            "type": row['type']
        })

    message_data = {
        "messages": messages_payload,
        "allowDuplicates": False  # [안전장치] 실제 운영 시 중복 스팸 발송 차단!
    }

    print(f"📱 [1/2단계] 라이브 고객 {len(messages_payload)}명 대상 CRM 문자 발송 시작...")
    try:
        solapi_result = send_message(API_KEY, API_SECRET, message_data)
        print(f" -> ✅ 솔라피 라이브 발송 완료! (Group ID: {solapi_result['groupInfo']['groupId']})")

        # 2. 빅쿼리 실제 로그 적재
        print(f"\n📊 [2/2단계] 빅쿼리 실시간 로그 적재 시작 ({BQ_TARGET_TABLE})...")

        # 상단에서 로드한 GCS 키 딕셔너리 활용
        bq_credentials = service_account.Credentials.from_service_account_info(key_file_dict)

        df_bq_load.to_gbq(  # df_bq_load 기반 빅쿼리 라이브 누적 적재
            destination_table=BQ_TARGET_TABLE,
            project_id=BQ_PROJECT_ID,
            credentials=bq_credentials,
            if_exists='append'
        )
        print(f" -> 🎉 [대성공] 빅쿼리 적재 완료! 총 {len(df_bq_load)}건의 실제 마케팅 발송 이력이 기록되었습니다.")

    except Exception as e:
        print(f"\n❌ 운영 파이프라인 가동 중 치명적 에러 발생: {e}")